# Risk Zone Identification
Phase 9: Deriving Low / Moderate / High risk zones from DBSCAN clusters

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

BASE_DIR = os.path.abspath('..')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

# Load clustered data (output of clustering.ipynb)
clustered_path = os.path.join(OUTPUT_DIR, 'clustered_earthquake_data.csv')
cleaned_path   = os.path.join(OUTPUT_DIR, 'cleaned_earthquake_data.csv')
data_path = clustered_path if os.path.exists(clustered_path) else cleaned_path

df = pd.read_csv(data_path, parse_dates=['Datetime'])

# If DBSCAN_Cluster column missing, re-run clustering here
if 'DBSCAN_Cluster' not in df.columns:
    from sklearn.cluster import DBSCAN
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    cs = scaler.fit_transform(df[['Latitude','Longitude']].values)
    df['DBSCAN_Cluster'] = DBSCAN(eps=0.2, min_samples=10).fit_predict(cs)
    print('Re-ran DBSCAN clustering.')

print(f'Loaded {len(df)} records. Clusters: {df["DBSCAN_Cluster"].unique()}')

## Aggregate Cluster Statistics

In [ ]:
cluster_df = df[df['DBSCAN_Cluster'] != -1]
risk = cluster_df.groupby('DBSCAN_Cluster').agg(
    Event_Count=('Magnitude','count'),
    Avg_Magnitude=('Magnitude','mean'),
    Max_Magnitude=('Magnitude','max'),
    Avg_Depth=('Depth','mean'),
    Center_Lat=('Latitude','mean'),
    Center_Lon=('Longitude','mean')
).reset_index().round(2)

q75 = risk['Event_Count'].quantile(0.75)
q50 = risk['Event_Count'].median()

def assign_risk(r):
    if r['Event_Count'] > q75 or r['Max_Magnitude'] >= 6.0:  return 'High Risk'
    elif r['Event_Count'] > q50 or r['Avg_Magnitude'] >= 4.5: return 'Moderate Risk'
    return 'Low Risk'

risk['Risk_Level'] = risk.apply(assign_risk, axis=1)
display(risk)

## Risk Zone Map

In [ ]:
color_map = {'High Risk': '#F44336', 'Moderate Risk': '#FF9800', 'Low Risk': '#4CAF50'}
plt.figure(figsize=(13, 9))

noise = df[df['DBSCAN_Cluster'] == -1]
plt.scatter(noise['Longitude'], noise['Latitude'], c='lightgrey', s=5, alpha=0.3, label='Noise/Scattered')

for _, row in risk.iterrows():
    cdata = df[df['DBSCAN_Cluster'] == row['DBSCAN_Cluster']]
    c = color_map[row['Risk_Level']]
    plt.scatter(cdata['Longitude'], cdata['Latitude'], c=c, s=15, alpha=0.65,
                label=f"Cluster {int(row['DBSCAN_Cluster'])} — {row['Risk_Level']} ({int(row['Event_Count'])} events)")

plt.title('Seismic Risk Zones in Nepal (1990–2026)', fontsize=14)
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.legend(loc='lower right', fontsize=9)
plt.savefig(os.path.join(OUTPUT_DIR, 'risk_zones_map.png'), dpi=150)
plt.show()

## Save Results

In [ ]:
risk.to_csv(os.path.join(OUTPUT_DIR, 'risk_zones_summary.csv'), index=False)

with open(os.path.join(OUTPUT_DIR, 'risk_zones_report.txt'), 'w', encoding='utf-8') as f:
    f.write('=== Risk Zone Identification ===\n\n')
    f.write('Methodology:\n')
    f.write('- High Risk: Event count > 75th percentile OR Max Magnitude >= 6.0\n')
    f.write('- Moderate Risk: Event count > median OR Avg Magnitude >= 4.5\n')
    f.write('- Low Risk: All other clusters\n\n')
    f.write('Risk Zone Breakdown:\n')
    f.write(risk.to_string())

print('Risk zone report and CSV saved to outputs/')